In [3]:
import asyncio
from playwright.async_api import async_playwright
import json

async def save_cookies():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        # Navigate to DeepSeek and manually log in (first run)
        await page.goto("https://chat.deepseek.com")
        input("Please log in manually, then press Enter...")  # Pause for login

        # Save cookies to a file
        cookies = await context.cookies()
        with open("deepseek_cookies.json", "w") as f:
            json.dump(cookies, f)
        
        await browser.close()

async def automate_with_persisted_session(question):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()

        # Load saved cookies
        with open("deepseek_cookies.json", "r") as f:
            cookies = json.load(f)
        await context.add_cookies(cookies)

        page = await context.new_page()
        await page.goto("https://chat.deepseek.com")

        # Check if still logged in
        if "登录" in await page.title():
            raise Exception("Session expired. Re-run `save_cookies()`.")

        # Submit question and fetch answer
        await page.fill("textarea[placeholder='输入你的问题...']", question)
        await page.click("button:has-text('发送')")
        await page.wait_for_selector(".answer-text", timeout=30000)
        answer = await page.inner_text(".answer-text")

        print("Answer:", answer)
        await browser.close()

# Run the async functions
async def main():
    # Uncomment to save cookies (first run only):
    # await save_cookies()  

    # Now automate questions in the same session
    await automate_with_persisted_session("How to send HTTP requests in Python?")
    await automate_with_persisted_session("Show me an example with `requests`.")

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop